## Import Libraries

In [3]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import os
import nltk
from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize
from nltk.corpus import cmudict
from nltk.corpus import stopwords
import string
from collections import defaultdict
nltk.download('punkt')
nltk.download('cmudict')
import re
import gensim
from nltk import sent_tokenize
from gensim.utils import simple_preprocess
from tqdm import tqdm
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from gensim.models import KeyedVectors


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/rajdipingale/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package cmudict to
[nltk_data]     /Users/rajdipingale/nltk_data...
[nltk_data]   Package cmudict is already up-to-date!


# Part 1
# Creating labeled Dataset using Lexicon Sentiment Analysis

## Extract Articles from Internet

In [4]:
def extract_article(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Assuming the article title is within the <title> tag
    title = soup.title.string if soup.title else 'No Title'

    # Extract the article text
    # Here we assume that the main article content is within <article> tag
    # This might need to be adjusted depending on the website's structure
    article_text = ''
    article = soup.find('article')
    if article:
        paragraphs = article.find_all('p')
        for paragraph in paragraphs:
            article_text += paragraph.get_text() + '\n'
    else:
        # Fallback: Extract all text from <p> tags
        paragraphs = soup.find_all('p')
        for paragraph in paragraphs:
            article_text += paragraph.get_text() + '\n'
    
    return title, article_text

In [6]:
# Read the Excel file
df = pd.read_excel('/Users/rajdipingale/Desktop/Resume Projects/Sentiment Analysis and Prediction from Web Articles/Input.xlsx')


In [7]:
# Iterate over each row in the DataFrame
df['Extracted_Article'] = None  # Initialize the column with None or empty strings

# Loop through each row in the DataFrame
for index, row in df.iterrows():
    url_id = row['URL_ID']
    url = row['URL']
    
    try:
        # Extract the title and article text from the URL
        title, article_text = extract_article(url)
        
        # Combine title and article text
        full_article = title + '\n\n' + article_text
        
        # Save the combined text in the new DataFrame column
        df.at[index, 'Extracted_Article'] = full_article
        
        print(f'Successfully extracted and saved article {url_id}')
    except Exception as e:
        print(f'Failed to extract article {url_id} from {url}: {e}')



Successfully extracted and saved article bctech2011
Successfully extracted and saved article bctech2012
Successfully extracted and saved article bctech2013
Successfully extracted and saved article bctech2014
Successfully extracted and saved article bctech2015
Successfully extracted and saved article bctech2016
Successfully extracted and saved article bctech2017
Successfully extracted and saved article bctech2018
Successfully extracted and saved article bctech2019
Successfully extracted and saved article bctech2020
Successfully extracted and saved article bctech2021
Successfully extracted and saved article bctech2022
Successfully extracted and saved article bctech2023
Successfully extracted and saved article bctech2024
Successfully extracted and saved article bctech2025
Successfully extracted and saved article bctech2026
Successfully extracted and saved article bctech2027
Successfully extracted and saved article bctech2028
Successfully extracted and saved article bctech2029
Successfully

## Sentiment Analysis
### Remove stopwords


In [8]:
# Function to load stopwords from text files
def load_stopwords(stopwords_dir, encoding='utf-8'):
    stopwords = set()
    for filename in os.listdir(stopwords_dir):
        file_path = os.path.join(stopwords_dir, filename)
        try:
            with open(file_path, 'r', encoding=encoding, errors='ignore') as file:
                for line in file:
                    stopwords.add(line.strip().lower())
        except UnicodeDecodeError as e:
            print(f"Error reading {file_path}: {e}")
    return stopwords

In [9]:
# Function to remove stopwords from text
def remove_stopwords(text, stopwords):
    words = text.split()
    cleaned_text = ' '.join([word for word in words if word.lower() not in stopwords])
    return cleaned_text


In [10]:
# Directory containing the stopword text files
stopwords_dir = '/Users/rajdipingale/Desktop/Resume Projects/Sentiment Analysis and Prediction from Web Articles/StopWords'

# Load the stopwords
stopwords = load_stopwords(stopwords_dir, encoding='utf-8')


In [11]:
# Create a new column to store the cleaned article text
df['Cleaned_Article'] = None

# Iterate over each row in the DataFrame
for index, row in df.iterrows():
    text = row['Extracted_Article']
    
    # Remove stopwords from the text
    cleaned_text = remove_stopwords(text, stopwords)
    
    # Save the cleaned text back into the DataFrame
    df.at[index, 'Cleaned_Article'] = cleaned_text


In [12]:
def load_and_filter_words(master_dict_file, stopwords, encoding='utf-8'):
    words = set()
    with open(master_dict_file, 'r', encoding=encoding, errors='ignore') as file:
        for line in file:
            word = line.strip().lower()
            if word and word not in stopwords:
                words.add(word)
    return words


### Calculate Polarity score

In [13]:
positive_words_file = '/Users/rajdipingale/Desktop/Resume Projects/Sentiment Analysis and Prediction from Web Articles/MasterDictionary/positive-words.txt'
negative_words_file = '/Users/rajdipingale/Desktop/Resume Projects/Sentiment Analysis and Prediction from Web Articles/MasterDictionary/negative-words.txt'


In [14]:
# Load and filter positive and negative words
positive_words = list(load_and_filter_words(positive_words_file, stopwords, encoding='utf-8'))
negative_words = list(load_and_filter_words(negative_words_file, stopwords, encoding='utf-8'))


In [ ]:
print("Positive Words:", positive_words)
print("Negative Words:", negative_words)

In [17]:

# Function to calculate the scores
def calculate_scores(text, positive_words, negative_words):
    tokens = word_tokenize(text.lower())
    positive_score = sum(1 for word in tokens if word in positive_words)
    negative_score = sum(1 for word in tokens if word in negative_words)
    polarity_score = (positive_score - negative_score) / ((positive_score + negative_score) + 0.000001)
    return polarity_score



In [18]:
# Calculate the polarity scores and add them to the DataFrame
df['Polarity Score'] = df['Cleaned_Article'].apply(lambda x: calculate_scores(x, positive_words, negative_words))


In [19]:
df.head()

,URL_ID,URL,Extracted_Article,Cleaned_Article,Polarity Score
0,bctech2011,https://insights.blackcoffer.com/ml-and-ai-bas...,ML and AI-based insurance premium model to pre...,ML AI-based insurance premium model predict pr...,0.619048
1,bctech2012,https://insights.blackcoffer.com/streamlined-i...,Streamlined Integration: Interactive Brokers A...,Streamlined Integration: Interactive Brokers A...,1.000000
2,bctech2013,https://insights.blackcoffer.com/efficient-dat...,Efficient Data Integration and User-Friendly I...,Efficient Data Integration User-Friendly Inter...,1.000000
3,bctech2014,https://insights.blackcoffer.com/effective-man...,Effective Management of Social Media Data Extr...,Effective Management Social Media Data Extract...,1.000000
4,bctech2015,https://insights.blackcoffer.com/streamlined-t...,Streamlined Trading Operations Interface for M...,Streamlined Trading Operations Interface MetaT...,1.000000


# Part 2
# Training Word2Vec Model

### Load the pretrained Word2Vec model


In [21]:
# You need to download the model separately from https://code.google.com/archive/p/word2vec/
model_path = '/Users/rajdipingale/gensim-data/word2vec-google-news-300/word2vec-google-news-300'
pretrained_model = KeyedVectors.load_word2vec_format(model_path, binary=True)

# Function to compute document embedding
def document_vector(doc):
    """Create a document vector by averaging word vectors."""
    words = doc.split()  # Tokenize the document
    word_vectors = [
        pretrained_model[word] for word in words if word in pretrained_model
    ]
    if not word_vectors:  
        return np.zeros(pretrained_model.vector_size)
    return np.mean(word_vectors, axis=0)



In [22]:
document_vector(df['Cleaned_Article'].values[0])


array([-4.40553166e-02,  1.78353284e-02,  1.04353402e-03,  3.09365094e-02,
       -6.07716292e-03, -1.95453945e-03,  6.73779994e-02, -3.27367447e-02,
        1.25595137e-01,  2.21451316e-02, -3.72983515e-02, -3.16230170e-02,
       -1.17378775e-02,  1.88897494e-02, -1.37819499e-01,  4.02693711e-02,
        1.36863645e-02,  9.44957063e-02, -4.78339233e-02, -6.12445138e-02,
       -1.37680564e-02,  1.46133574e-02, -5.09792082e-02,  9.61954966e-02,
        3.24842595e-02,  6.63861707e-02, -5.61399013e-02,  8.10553506e-02,
       -1.33293774e-02, -5.56790009e-02, -7.94124883e-03, -5.09553961e-02,
        2.92990170e-03, -2.94895489e-02,  3.71371917e-02, -2.83981878e-02,
        4.76993397e-02, -1.45004783e-02,  3.38600352e-02, -5.61615592e-03,
        1.09020406e-02,  4.02331203e-02, -1.64690558e-02,  2.94076949e-02,
       -7.51917586e-02, -1.63616464e-01, -6.77377731e-02, -1.77854905e-03,
       -1.56029686e-03,  3.57867852e-02, -1.72650311e-02, -9.23337787e-03,
       -7.64678344e-02, -

In [23]:
X = []
for doc in tqdm(df['Cleaned_Article'].values):
    X.append(document_vector(doc))

100%|██████████| 147/147 [00:00<00:00, 494.08it/s]


In [24]:
X = np.array(X)
X[0]


array([-4.40553166e-02,  1.78353284e-02,  1.04353402e-03,  3.09365094e-02,
       -6.07716292e-03, -1.95453945e-03,  6.73779994e-02, -3.27367447e-02,
        1.25595137e-01,  2.21451316e-02, -3.72983515e-02, -3.16230170e-02,
       -1.17378775e-02,  1.88897494e-02, -1.37819499e-01,  4.02693711e-02,
        1.36863645e-02,  9.44957063e-02, -4.78339233e-02, -6.12445138e-02,
       -1.37680564e-02,  1.46133574e-02, -5.09792082e-02,  9.61954966e-02,
        3.24842595e-02,  6.63861707e-02, -5.61399013e-02,  8.10553506e-02,
       -1.33293774e-02, -5.56790009e-02, -7.94124883e-03, -5.09553961e-02,
        2.92990170e-03, -2.94895489e-02,  3.71371917e-02, -2.83981878e-02,
        4.76993397e-02, -1.45004783e-02,  3.38600352e-02, -5.61615592e-03,
        1.09020406e-02,  4.02331203e-02, -1.64690558e-02,  2.94076949e-02,
       -7.51917586e-02, -1.63616464e-01, -6.77377731e-02, -1.77854905e-03,
       -1.56029686e-03,  3.57867852e-02, -1.72650311e-02, -9.23337787e-03,
       -7.64678344e-02, -

In [25]:
y=df['Polarity Score']

## Train ML Models

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [27]:
def evaluation_metrics(actual, pred):
    MAE = mean_absolute_error(actual, pred)
    MSE = mean_squared_error(actual, pred)
    RMSE = np.sqrt(mean_squared_error(actual, pred))
    SCORE = r2_score(actual, pred)
    return print(f"R2 Score: {SCORE:.4f} \nMean Absolute Error:{MAE:.4f},\nMean Squared Error:{MSE:.4f} \nRoot Mean Squared Error:{RMSE:.4f}")

In [ ]:
model_lr = LinearRegression()
prediction_lr = model_lr.fit(X_train, y_train).predict(X_test)
evaluation_metrics(y_test, prediction_lr)

In [ ]:
model_svm = SVR()
prediction_svm = model_svm.fit(X_train, y_train).predict(X_test)
evaluation_metrics(y_test, prediction_svm)

In [ ]:
model_xgb = XGBRegressor()
prediction_xgb = model_xgb.fit(X_train, y_train).predict(X_test)
evaluation_metrics(y_test, prediction_xgb)